In [1]:
import io
import zipfile
import requests
import frontmatter

print("Setup successful")

Setup successful


In [2]:
import io
import zipfile
import requests
import frontmatter


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse markdown files
    from a GitHub repository.
    """

    prefix = "https://codeload.github.com"

    url = (
        f"{prefix}/{repo_owner}/"
        f"{repo_name}/zip/refs/heads/main"
    )

    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(
            f"Failed to download repository: "
            f"{resp.status_code}"
        )

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():

        filename = file_info.filename
        filename_lower = filename.lower()

        if not (
            filename_lower.endswith(".md")
            or filename_lower.endswith(".mdx")
        ):
            continue

        try:
            with zf.open(file_info) as f_in:

                content = (
                    f_in.read()
                    .decode("utf-8", errors="ignore")
                )

                post = frontmatter.loads(content)

                data = post.to_dict()

                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(
                f"Error processing "
                f"{filename}: {e}"
            )

    zf.close()

    return repository_data

In [3]:
data = read_repo_data(
    "DataTalksClub",
    "faq"
)

len(data)

1216

In [4]:
data[0]

{'description': 'Review and process open FAQ PRs',
 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -

In [5]:
data[:5]

[{'description': 'Review and process open FAQ PRs',
  'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep

In [6]:
len(data)

1216

In [7]:
data[10]

{'id': 'a83f047f52',
 'question': 'How to troubleshoot issues and ask good questions',
 'sort_order': 6,
 'content': "**Try to solve it yourself first**\n\n- Read the error message carefully — it usually includes a line number, a stack trace, and a description of what went wrong.\n- Search the message: copy the most specific part of the error (not the whole stack trace) into Google. The format `<tool> <error message>` works well, e.g. `pgcli error column c.relhasoids does not exist`.\n- Check the official documentation of the tool you're using.\n- Use Ctrl+F in this FAQ and in Slack channel pinned messages.\n- Restart the process / container / shell / VM and try once more — many transient errors resolve this way.\n- If you suspect the install is broken, uninstall first, then reinstall. Reinstalling on top of a broken install rarely helps.\n\n**Asking for help in Slack / forums**\n\nWhen the troubleshooting steps don't help and you need another pair of eyes, include enough info that som